# LangGraph — Simple Agent

**Framework:** LangGraph  
**Level:** 01 — Simple  
**Model:** `gemini-2.0-flash` via `langchain-google-genai`

### What You Will Learn
- How LangGraph models an agent as a **state machine** (graph of nodes + edges)
- How to define typed state with an `add_messages` reducer
- How `ToolNode` and `tools_condition` automate tool execution and routing
- Why LangGraph makes the ReAct loop **explicit** — you see every edge and node

## Concept: The State Machine

```
              START
                │
                ▼
┌───────────────────────────────┐
│          llm_node             │
│  Reads messages from state    │
│  Calls LLM with bound tools   │
│  Appends response to state    │
└───────────────┬───────────────┘
                │
        tools_condition
        ┌───────┴───────┐
  tool_calls?       no tool_calls?
        │                   │
        ▼                   ▼
┌──────────────┐           END
│  tools_node  │
│  Executes    │
│  tool calls  │
│  Appends     │
│  results     │
└──────┬───────┘
       │
       └──────────→ back to llm_node
```

**Key insight:** Unlike LangChain's `AgentExecutor` (which hides the loop), LangGraph forces you to **draw the loop explicitly** as graph edges. This makes it:
- Easier to inspect and debug (every state transition is visible)
- Easier to customize (add a human-approval node, a retry node, etc.)
- The foundation for complex multi-agent architectures

**State** is a typed dict that accumulates messages. The `add_messages` reducer means new messages are **appended** to the list, not overwritten.

## Setup

In [ ]:
import os
from typing import Annotated
from typing_extensions import TypedDict
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "GOOGLE_API_KEY not set — check your .env file"
print("✓ GOOGLE_API_KEY loaded")

## Tool Definitions

Same `@tool` decorator as LangChain — LangGraph uses LangChain's tool layer.  
The difference comes later, in how tools are **executed and routed** inside the graph.

In [ ]:
@tool
def get_weather(city: str) -> dict:
    """Return a mock weather report for the given city."""
    mock_data = {
        "london":    {"condition": "Cloudy", "temp_c": 12, "humidity": 78},
        "new york":  {"condition": "Sunny",  "temp_c": 22, "humidity": 45},
        "bangalore": {"condition": "Rainy",  "temp_c": 25, "humidity": 85},
        "tokyo":     {"condition": "Clear",  "temp_c": 18, "humidity": 60},
    }
    key = city.lower()
    if key in mock_data:
        d = mock_data[key]
        return {"city": city, "condition": d["condition"],
                "temperature_celsius": d["temp_c"], "humidity_percent": d["humidity"]}
    return {"error": f"No data for '{city}'. Try: London, New York, Bangalore, Tokyo."}


@tool
def get_time(city: str) -> dict:
    """Return the current local time for the given city."""
    mock_times = {
        "london":    "14:30 GMT",
        "new york":  "09:30 EST",
        "bangalore": "20:00 IST",
        "tokyo":     "22:30 JST",
    }
    key = city.lower()
    if key in mock_times:
        return {"city": city, "local_time": mock_times[key]}
    return {"error": f"No time data for '{city}'. Try: London, New York, Bangalore, Tokyo."}


tools = [get_weather, get_time]
print(f"Defined {len(tools)} tools: {[t.name for t in tools]}")

## State Definition

State is the single source of truth passed between nodes.  
`Annotated[list, add_messages]` means: this field is a list, and when updating it, **append** new items rather than replace.

In [ ]:
class AgentState(TypedDict):
    # Every message (human, AI, tool call, tool result) accumulates here
    messages: Annotated[list, add_messages]

print("AgentState defined with messages field (add_messages reducer)")

## Model + Nodes

**`llm_node`** — reads the message history from state, calls the LLM, appends the response.  
**`tools_node`** — `ToolNode` is a prebuilt node that reads tool_calls from the last message, executes them, and appends results.

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
llm_with_tools = llm.bind_tools(tools)  # tells the LLM about available tools

SYSTEM_MSG = SystemMessage(content=(
    "You are a helpful travel assistant. "
    "Use tools to answer weather and time questions. Be concise and friendly."
))

def llm_node(state: AgentState) -> dict:
    """Call the LLM. Prepend system message, pass all current messages."""
    messages = [SYSTEM_MSG] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}  # add_messages appends this to the state list


tools_node = ToolNode(tools)  # prebuilt: reads tool_calls from last AI message, executes, returns ToolMessage

print("Nodes defined: llm_node, tools_node")

## Build the Graph

This is where LangGraph is unique: you **draw the agent as a graph**.  
`tools_condition` is a prebuilt conditional function that returns `"tools"` if the last message has `tool_calls`, or `END` otherwise.

In [ ]:
builder = StateGraph(AgentState)

# Add nodes
builder.add_node("llm", llm_node)
builder.add_node("tools", tools_node)

# Entry point
builder.set_entry_point("llm")

# Conditional edge from llm: tools or END
builder.add_conditional_edges("llm", tools_condition)

# After tools run, always go back to LLM
builder.add_edge("tools", "llm")

graph = builder.compile()
print("Graph compiled")

# Optional: visualize the graph structure
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print("(Graph visualization skipped — install graphviz to enable)")

## Run the Agent

In [ ]:
def run(query: str) -> str:
    result = graph.invoke({"messages": [HumanMessage(content=query)]})
    return result["messages"][-1].content  # last message = final AI response


# Query 1 — single tool call
query1 = "What's the weather like in Bangalore right now?"
print(f"User: {query1}")
print(f"Agent: {run(query1)}")

In [ ]:
# Query 2 — two tool calls
query2 = "What time is it in Tokyo, and is it a good day to go outside?"
print(f"User: {query2}")
print(f"Agent: {run(query2)}")

In [ ]:
# Inspect the full message trace — shows every step of the loop
query3 = "What's the weather in London and what time is it there?"
result = graph.invoke({"messages": [HumanMessage(content=query3)]})

print("Full message trace:")
for i, msg in enumerate(result["messages"]):
    msg_type = type(msg).__name__
    content = msg.content if isinstance(msg.content, str) else str(msg.content)[:120]
    print(f"  [{i}] {msg_type}: {content[:100]}")

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `AgentState` with `add_messages` | Typed state; messages accumulate rather than overwrite |
| `llm_node` as a plain function | Any Python function can be a node — maximum flexibility |
| `ToolNode(tools)` | Prebuilt node handles all tool execution boilerplate |
| `tools_condition` | Prebuilt conditional: routes based on whether tool_calls exist |
| `builder.add_conditional_edges` | The ReAct loop is a first-class graph edge — explicit and inspectable |

### LangGraph vs LangChain vs ADK (Simple Level)

| Dimension | ADK | LangChain | LangGraph |
|---|---|---|---|
| Loop visibility | Hidden in runner | Hidden in executor | **Explicit** as graph edges |
| State | Session object | Message list | Typed `AgentState` dict |
| Tool execution | ADK runtime | `AgentExecutor` | `ToolNode` |
| Customization | Agent config | LCEL chain | Add/modify nodes and edges |
| Best for | GCP production | Broad ecosystem | Complex flows + multi-agent |

### What Changes at 02-Intermediate
- State gains more fields (not just messages) — e.g., `user_profile`, `task_status`
- Adds a **human-in-the-loop** interrupt node
- Adds `MemorySaver` checkpointer for persistent state across calls

### Next Notebook
[02-Intermediate →](../02-intermediate/agent.ipynb)